# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Baseline Rule

A webpage should be prioritized for content refresh if it receives many impressions but has a low click-through rate and a poor average search position. These pages already have search visibility but are not attracting enough clicks, making them good candidates for content improvement.

### Signals Used

- High impressions
- Low CTR
- Poor average position

### Reason Codes

- **HIGH_IMPRESSIONS** – The webpage receives strong search visibility.
- **LOW_CTR** – Users see the page but do not click frequently.
- **POOR_POSITION** – The webpage ranks lower in search results and may benefit from optimization.

In [4]:
import pandas as pd

# Load dataset
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

print("=" * 50)
print("SIGNAL SUMMARY")
print("=" * 50)

print("Average Impressions:", round(df["impressions_90d"].mean(),2))
print("Average CTR:", round(df["ctr"].mean(),2))
print("Average Position:", round(df["avg_position"].mean(),2))

display(df[["impressions_90d","ctr","avg_position"]].describe())

SIGNAL SUMMARY
Average Impressions: 5200.37
Average CTR: 0.51
Average Position: 16.34


,impressions_90d,ctr,avg_position
count,30000.000000,30000.000000,30000.00000
mean,5200.366300,0.510733,16.34238
std,16838.019547,3.279162,15.21679
min,1.000000,0.000000,0.00000
25%,81.000000,0.000000,6.20000
50%,731.000000,0.070000,10.80000
75%,3615.250000,0.290000,22.30000
max,517715.000000,100.000000,245.00000


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [5]:
import os
import pandas as pd

# Load dataset
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# -------------------------------
# Thresholds
# -------------------------------
impression_threshold = df["impressions_90d"].mean()
ctr_threshold = df["ctr"].mean()
position_threshold = 20

# -------------------------------
# Baseline Action Score
# -------------------------------
df["action_score"] = (
    (df["impressions_90d"] > impression_threshold).astype(int)
    + (df["ctr"] < ctr_threshold).astype(int)
    + (df["avg_position"] > position_threshold).astype(int)
)

# -------------------------------
# Reason Code
# -------------------------------
def reason_code(row):
    reasons = []

    if row["impressions_90d"] > impression_threshold:
        reasons.append("HIGH_IMPRESSIONS")

    if row["ctr"] < ctr_threshold:
        reasons.append("LOW_CTR")

    if row["avg_position"] > position_threshold:
        reasons.append("POOR_POSITION")

    if len(reasons) == 0:
        return "NONE"

    return "; ".join(reasons)

df["reason_code"] = df.apply(reason_code, axis=1)

# -------------------------------
# Action Label
# -------------------------------
def action_label(score):
    if score == 3:
        return "Refresh Immediately"
    elif score == 2:
        return "Review Soon"
    elif score == 1:
        return "Monitor"
    else:
        return "No Action"

df["action_label"] = df["action_score"].apply(action_label)

# -------------------------------
# Rank
# -------------------------------
ranked_df = (
    df.sort_values(
        by=["action_score", "impressions_90d"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

# -------------------------------
# Save CSV
# -------------------------------
os.makedirs("../../work/outputs", exist_ok=True)

output_path = "../../work/outputs/baseline_action_score.csv"

ranked_df.to_csv(output_path, index=False)

print("=" * 50)
print("Baseline Action Queue Created")
print("=" * 50)

print(f"Saved to: {output_path}")

display(
    ranked_df[
        [
            "content_id",
            "action_score",
            "reason_code",
            "action_label"
        ]
    ].head(10)
)

Baseline Action Queue Created
Saved to: ../../work/outputs/baseline_action_score.csv


,content_id,action_score,reason_code,action_label
0,content_2cb567c3c89b,3,HIGH_IMPRESSIONS; LOW_CTR; POOR_POSITION,Refresh Immediately
1,content_2dba2b1f9536,3,HIGH_IMPRESSIONS; LOW_CTR; POOR_POSITION,Refresh Immediately
2,content_b28d1efd668f,3,HIGH_IMPRESSIONS; LOW_CTR; POOR_POSITION,Refresh Immediately
3,content_813e88069237,3,HIGH_IMPRESSIONS; LOW_CTR; POOR_POSITION,Refresh Immediately
4,content_ff94c9b6b411,3,HIGH_IMPRESSIONS; LOW_CTR; POOR_POSITION,Refresh Immediately
5,content_66b4046cc144,3,HIGH_IMPRESSIONS; LOW_CTR; POOR_POSITION,Refresh Immediately
6,content_a023517539fe,3,HIGH_IMPRESSIONS; LOW_CTR; POOR_POSITION,Refresh Immediately
7,content_b511d4bc4ad2,3,HIGH_IMPRESSIONS; LOW_CTR; POOR_POSITION,Refresh Immediately
8,content_f02b48f88241,3,HIGH_IMPRESSIONS; LOW_CTR; POOR_POSITION,Refresh Immediately
9,content_05e9b4cd9ccf,3,HIGH_IMPRESSIONS; LOW_CTR; POOR_POSITION,Refresh Immediately


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [6]:
print("=" * 60)
print("TOP 20 REVIEW")
print("=" * 60)

top20 = ranked_df.head(20).copy()

top20["confidence"] = top20["action_score"].map({
    3: "High",
    2: "Medium",
    1: "Low",
    0: "Very Low"
})

top20["what_could_make_it_wrong"] = (
    "Recent content updates, seasonal traffic changes, "
    "or missing information not captured in the dataset."
)

review_columns = [
    "content_id",
    "action_score",
    "action_label",
    "reason_code",
    "confidence",
    "what_could_make_it_wrong"
]

display(top20[review_columns])

TOP 20 REVIEW


,content_id,action_score,action_label,reason_code,confidence,what_could_make_it_wrong
0,content_2cb567c3c89b,3,Refresh Immediately,HIGH_IMPRESSIONS; LOW_CTR; POOR_POSITION,High,"Recent content updates, seasonal traffic chang..."
1,content_2dba2b1f9536,3,Refresh Immediately,HIGH_IMPRESSIONS; LOW_CTR; POOR_POSITION,High,"Recent content updates, seasonal traffic chang..."
2,content_b28d1efd668f,3,Refresh Immediately,HIGH_IMPRESSIONS; LOW_CTR; POOR_POSITION,High,"Recent content updates, seasonal traffic chang..."
3,content_813e88069237,3,Refresh Immediately,HIGH_IMPRESSIONS; LOW_CTR; POOR_POSITION,High,"Recent content updates, seasonal traffic chang..."
4,content_ff94c9b6b411,3,Refresh Immediately,HIGH_IMPRESSIONS; LOW_CTR; POOR_POSITION,High,"Recent content updates, seasonal traffic chang..."
5,content_66b4046cc144,3,Refresh Immediately,HIGH_IMPRESSIONS; LOW_CTR; POOR_POSITION,High,"Recent content updates, seasonal traffic chang..."
6,content_a023517539fe,3,Refresh Immediately,HIGH_IMPRESSIONS; LOW_CTR; POOR_POSITION,High,"Recent content updates, seasonal traffic chang..."
7,content_b511d4bc4ad2,3,Refresh Immediately,HIGH_IMPRESSIONS; LOW_CTR; POOR_POSITION,High,"Recent content updates, seasonal traffic chang..."
8,content_f02b48f88241,3,Refresh Immediately,HIGH_IMPRESSIONS; LOW_CTR; POOR_POSITION,High,"Recent content updates, seasonal traffic chang..."
9,content_05e9b4cd9ccf,3,Refresh Immediately,HIGH_IMPRESSIONS; LOW_CTR; POOR_POSITION,High,"Recent content updates, seasonal traffic chang..."


### Top-20 Review

The top 20 webpages were selected based on the baseline action score. Each recommendation includes an action label, a reason code, and a confidence level to help content teams prioritize their work.

### Review Summary

- **Action** indicates the recommended next step for each webpage.
- **Reason Code** explains why the webpage received its score.
- **Confidence** increases with the action score.
- **What Could Make It Wrong** reminds reviewers that historical metrics cannot capture every real-world factor, such as recent content updates, seasonality, or external search engine changes.

This review provides transparent and explainable recommendations that can be manually verified before taking action.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak Picks

Some webpages in the top-ranked list may still be weak recommendations because the baseline rule is intentionally simple. A page may receive a high action score due to high impressions, low CTR, and poor average position even if it was recently updated or affected by seasonal search trends. The baseline cannot detect these situations.

### Leakage Check

No future information was used to calculate the baseline action score. The rule only uses historical features available before making a refresh recommendation:

- impressions_90d
- ctr
- avg_position

The rule does not use any future performance metrics, manually assigned labels, or product-specific flags. Therefore, no data leakage was intentionally introduced into the baseline.

In [7]:
print("=" * 60)
print("WEAK PICKS & LEAKAGE CHECK")
print("=" * 60)

# Potential weak picks
weak_picks = ranked_df[
    (ranked_df["action_score"] >= 2) &
    (ranked_df["engagement_rate"] > ranked_df["engagement_rate"].median())
]

print(f"Potential weak picks found: {len(weak_picks)}")

display(
    weak_picks[
        [
            "content_id",
            "action_score",
            "engagement_rate",
            "reason_code",
            "action_label"
        ]
    ].head(10)
)

print("\nLeakage Check")
print("---------------------------")

features_used = [
    "impressions_90d",
    "ctr",
    "avg_position"
]

print("Features used in the baseline:")
for feature in features_used:
    print(f"- {feature}")

print("\nNo future-window columns or label-derived fields were used.")

WEAK PICKS & LEAKAGE CHECK
Potential weak picks found: 4335


,content_id,action_score,engagement_rate,reason_code,action_label
0,content_2cb567c3c89b,3,5.82,HIGH_IMPRESSIONS; LOW_CTR; POOR_POSITION,Refresh Immediately
1,content_2dba2b1f9536,3,2.73,HIGH_IMPRESSIONS; LOW_CTR; POOR_POSITION,Refresh Immediately
2,content_b28d1efd668f,3,3.83,HIGH_IMPRESSIONS; LOW_CTR; POOR_POSITION,Refresh Immediately
3,content_813e88069237,3,1.37,HIGH_IMPRESSIONS; LOW_CTR; POOR_POSITION,Refresh Immediately
4,content_ff94c9b6b411,3,1.15,HIGH_IMPRESSIONS; LOW_CTR; POOR_POSITION,Refresh Immediately
5,content_66b4046cc144,3,1.35,HIGH_IMPRESSIONS; LOW_CTR; POOR_POSITION,Refresh Immediately
6,content_a023517539fe,3,3.40,HIGH_IMPRESSIONS; LOW_CTR; POOR_POSITION,Refresh Immediately
7,content_b511d4bc4ad2,3,2.56,HIGH_IMPRESSIONS; LOW_CTR; POOR_POSITION,Refresh Immediately
8,content_f02b48f88241,3,9.19,HIGH_IMPRESSIONS; LOW_CTR; POOR_POSITION,Refresh Immediately
9,content_05e9b4cd9ccf,3,0.55,HIGH_IMPRESSIONS; LOW_CTR; POOR_POSITION,Refresh Immediately



Leakage Check
---------------------------
Features used in the baseline:
- impressions_90d
- ctr
- avg_position

No future-window columns or label-derived fields were used.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.